# New Patient Starts (NPS) Features — All Cohorts

Cohort-based refactor of `05_claim_rx_nps_monthly` (NBA 3.0, Wenjie Chen). Sections 1–3 walk through every table the original uses and what it builds from them; the cohort refactor and validation follow in later sections.

Source: `/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/03_Feature_Engineering/03_HCP_Level_Features/05_claim_rx_nps_monthly`

## What this notebook does, in plain words

It counts, per HCP per month, how many patients **started** a drug class or brand with that HCP.

A "start" is the patient's **first ever Rx claim** for that drug class (or brand) in the data, written by **any** HCP. The HCP who wrote that first claim gets credit for the start.

| Feature | Question it answers | Column |
| --- | --- | --- |
| Patients visiting | How many distinct patients had an Rx claim with this HCP this month? (denominator for `PCT_`) | `TOTAL_PATIENTS_VISIT` |
| Starts per drug class | How many patients started each `TREATMENT_CATEGORY` with this HCP? | `TOTAL_PATIENTS_START_<CLASS>` (e.g. `..._SGLT2`) |
| Starts per brand | Same for 8 brand groups (INVOKANA 100/300MG, FARXIGA, JARDIANCE, SPIRONOLACTONE-MRA, Diuretic + MRA, DPP-IV, GLP-1) | `TOTAL_PATIENTS_START_<BRAND>` (e.g. `..._JARDIANCE`) |
| Total new patients | How many distinct patients started *any* drug class with this HCP? | `TOTAL_NEW_PATIENT_START` |

Then the usual steps: fill missing months, sum over the last 1/3/6/12 months, `RUNNING_` totals, `PCT_` shares (each start column / `TOTAL_PATIENTS_VISIT` in the same window) and write one row per provider per month to `kerendia_nba_3_0_claim_rx_nps_monthly`.

Unlike notebooks 01–04, the input table already carries `PROVIDER_ID`, so the original works at provider level from the start (no NPI to provider mapping step).

**How to read this notebook:** section 1 shows the input table; section 2 shows the claim-level tables the original builds (the step just before counting); section 3 shows the original output. Everything is shown for the 10 test HCPs (validation mode).

## 0. Connection and Config

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "../00_cohort_config"

## 1. The input table (`kerendia_nba_3_0_Rx_05`)

The original reads a single table: one row per Rx claim, with the HCP as `PROVIDER_ID` (and `NPI_NUMBER`), the HCP's `PRIMARY_SPECIALTY_CODE`, the patient, the service date, `TREATMENT_CATEGORY` (drug class) and `CATEGORY` (brand group).

Two views are loaded:
* **`df_rx`**: claims written by the 10 test HCPs (what gets counted).
* **`df_rx_pat`**: *all* claims of those HCPs' patients, written by any HCP. The first-start date of a patient has to be looked up across every HCP, so this view is needed to reproduce the original exactly in validation mode.

The original keeps claims from 2019-11 onwards (cell 7). The table starts in 2021-01, so that filter removes nothing.

3.1 team rule: the counted claims (`df_rx_run`, built in the cell "Feature rows (once) and cohort list") start at `DATA_START_DATE` (2023-01-01), `RUNNING_` included. `df_rx_pat` keeps the full history, because a start is the patient's first-ever claim for the class or brand: a patient who started in 2021 must not look like a new start in 2023. The walkthrough and the offset-0 parity use the full history, as the original.

In [0]:
import builtins
# Paths, brands and the Rx history start come from 00_cohort_config (HCP shared constants)
_QUERY_OUTPUT    = HCP_QUERY_OUTPUT
_NOTEBOOK_OUTPUT = HCP_NOTEBOOK_OUTPUT
_BRANDS          = HCP_BRANDS

# Same cleaning and date filter as the original cell 7
df_rx_raw = spark.read.format("delta").load(_QUERY_OUTPUT + "kerendia_nba_3_0_Rx_05")
df_rx_all = (df_rx_raw
             .withColumn("TREATMENT_CATEGORY", F.regexp_replace("TREATMENT_CATEGORY", r"[\s]*$", ""))
             .filter(F.col(SVC_COL) >= HCP_RX_HISTORY_START))   # original 2019-11-01 filter

# Claims written by the HCPs of this run (validation mode: the 10 test HCPs), full history.
# DATA_START_DATE is applied to the counted claims only (df_rx_run, cell "Feature rows (once) and cohort list").
df_rx = apply_npi_filter(df_rx_all, NPI_COL)

# All claims of those HCPs' patients, by any HCP (needed for the first-start date)
_rx_patients = df_rx.select("PATIENT_ID").distinct()
df_rx_pat = df_rx_all.join(_rx_patients, on="PATIENT_ID", how="left_semi")

print(f"Rx claims of this run's HCPs: {df_rx.count():,} | patients: {_rx_patients.count():,} | "
      f"all claims of those patients (any HCP): {df_rx_pat.count():,}")

In [0]:
print(f"kerendia_nba_3_0_Rx_05: {len(df_rx_raw.columns)} columns")
display(df_rx_raw.agg(F.count(F.lit(1)).alias("rows"), F.min(SVC_COL).alias("first_date"),
                      F.max(SVC_COL).alias("last_date"), F.countDistinct("PROVIDER_ID").alias("providers")))
display(df_rx.orderBy(NPI_COL, SVC_COL).limit(50))

# One provider and one specialty per test HCP?
display(df_rx.groupBy(NPI_COL, "PROVIDER_ID")
        .agg(F.count(F.lit(1)).alias("claims"), F.countDistinct("PATIENT_ID").alias("patients"),
             F.countDistinct("PRIMARY_SPECIALTY_CODE").alias("specialty_codes"),
             F.first("PRIMARY_SPECIALTY_CODE").alias("specialty"))
        .orderBy(F.desc("claims")))

# Drug classes, and brand groups used by the brand features
display(df_rx.groupBy("TREATMENT_CATEGORY").count().orderBy(F.desc("count")))
display(df_rx.filter(F.col("CATEGORY").isin(_BRANDS)).groupBy("CATEGORY").count().orderBy(F.desc("count")))

## 2. What the original builds (claim level, before counting)

The claim month here is the cohort month from `00_cohort_config` (`assign_cohort_month`, offset `OFFSET_DAYS`); at offset 0 it equals the original calendar month (`date_trunc('MM', SVC_DT)`).

### 2.1 First-start dates (original cells 12 and 15)
For every patient and drug class, the date of the patient's **first** claim of that class in the whole table, by any HCP (`df_category_prep`). Same for every patient and brand group, for the 8 brands (`df_brand_prep`).

Because the table starts on 2021-01-02, "first ever" really means "first since 2021-01": patients who were already on a drug before 2021 look like new starts in early 2021. The output keeps months from 2022-01, so 2021 acts as a 12-month burn-in.

Tables below: sample first-start rows, and first-start dates per month (shows the early-2021 spike).

In [0]:
first_cat = (df_rx_pat.groupBy("PATIENT_ID", "TREATMENT_CATEGORY")
             .agg(F.min(SVC_COL).alias("FIRST_SVC_DT")))
first_brand = (df_rx_pat.filter(F.col("CATEGORY").isin(_BRANDS))
               .groupBy("PATIENT_ID", "CATEGORY")
               .agg(F.min(SVC_COL).alias("FIRST_SVC_DT")))

print(f"Patient x drug class first starts: {first_cat.count():,} | patient x brand first starts: {first_brand.count():,}")
display(first_cat.orderBy("PATIENT_ID", "FIRST_SVC_DT").limit(50))
display(first_cat.groupBy(F.date_trunc("MM", "FIRST_SVC_DT").alias("FIRST_START_MONTH")).count()
        .orderBy("FIRST_START_MONTH"))

### 2.2 Claims flagged as a start (original cells 12, 15 and 29)
The HCP's claims are made distinct on (provider, specialty, patient, date, drug class) and joined to the first-start dates on (patient, date, drug class). A claim on the patient's first date gets `NPS_FLAG = 1`; only those are counted. The brand version does the same on (patient, date, brand group).

Points to note:
* If two HCPs write the same patient's first claim on the same day, **both** get the start.
* The count is on distinct (provider, specialty, patient, date, class) rows, so a start is counted once even if the HCP wrote several claims that day.
* `TOTAL_NEW_PATIENT_START` = distinct patients with at least one drug-class start that month (brand starts are not used for it).

Tables below: claims before / after the flag, one patient's claims with the flag (example), and starts per drug class and per brand.

In [0]:
df_rx_m = df_rx.withColumn("SVC_DT_M", assign_cohort_month(F.col(SVC_COL)))

claims_cat = (
    df_rx_m.select("PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY", "SVC_DT_M").distinct()
    .join(first_cat.withColumnRenamed("FIRST_SVC_DT", SVC_COL).withColumn("NPS_FLAG", F.lit(1)),
          on=["PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY"], how="left")
)
claims_brand = (
    df_rx_m.select("PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "SVC_DT_M", "CATEGORY").distinct()
    .filter(F.col("CATEGORY").isin(_BRANDS))
    .join(first_brand.withColumnRenamed("FIRST_SVC_DT", SVC_COL).withColumn("NPS_FLAG", F.lit(1)),
          on=["PATIENT_ID", SVC_COL, "CATEGORY"], how="left")
)
_starts_cat   = claims_cat.filter("NPS_FLAG = 1")
_starts_brand = claims_brand.filter("NPS_FLAG = 1")
print(f"Drug class: distinct claim rows {claims_cat.count():,} | flagged as start {_starts_cat.count():,}")
print(f"Brand     : distinct claim rows {claims_brand.count():,} | flagged as start {_starts_brand.count():,}")

# Starts credited to more than one HCP (same patient, class and first date) among all claims of these patients
_shared = (df_rx_pat.select("PROVIDER_ID", "PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY").distinct()
           .join(first_cat.withColumnRenamed("FIRST_SVC_DT", SVC_COL), on=["PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY"], how="inner")
           .groupBy("PATIENT_ID", "TREATMENT_CATEGORY").agg(F.countDistinct("PROVIDER_ID").alias("hcps"))
           .groupBy("hcps").count().withColumnRenamed("count", "patient_class_starts").orderBy("hcps"))
display(_shared)

# Example: the patient with most claims for the first test HCP, with the flag
_pat = (df_rx.groupBy("PATIENT_ID").count().orderBy(F.desc("count")).first()["PATIENT_ID"])
display(claims_cat.filter(F.col("PATIENT_ID") == _pat).orderBy(SVC_COL, "TREATMENT_CATEGORY"))

display(_starts_cat.groupBy("TREATMENT_CATEGORY").count().orderBy(F.desc("count")))
display(_starts_brand.groupBy("CATEGORY").count().orderBy(F.desc("count")))

### 2.3 Monthly counts per provider (original cells 12, 17 and 29)
What the original counts per provider per month before filling months and adding windows:
* `TOTAL_PATIENTS_VISIT` = distinct patients with any claim (`df_apt`).
* `TOTAL_PATIENTS_START_<CLASS / BRAND>` = flagged rows per class / brand.
* `TOTAL_NEW_PATIENT_START` = distinct patients with a drug-class start.

The spine is the provider-months with a visit; start columns are left-joined and missing values set to 0. `TOTAL_NEW_PATIENT_START` is built and windowed separately and joined at the end without a fill, so it can be null (the original output has nulls in it).

Table below: these monthly counts for one test HCP.

In [0]:
_visits = (df_rx_m.select("PROVIDER_ID", "PATIENT_ID", "SVC_DT_M").distinct()
           .groupBy("PROVIDER_ID", "SVC_DT_M").agg(F.count(F.lit(1)).alias("TOTAL_PATIENTS_VISIT")))
_cat_m   = (_starts_cat.groupBy("PROVIDER_ID", "SVC_DT_M")
            .agg(F.count(F.lit(1)).alias("CLASS_STARTS"),
                 F.countDistinct("PATIENT_ID").alias("TOTAL_NEW_PATIENT_START")))
_brand_m = _starts_brand.groupBy("PROVIDER_ID", "SVC_DT_M").agg(F.count(F.lit(1)).alias("BRAND_STARTS"))
monthly = (_visits.join(_cat_m, ["PROVIDER_ID", "SVC_DT_M"], "left")
                  .join(_brand_m, ["PROVIDER_ID", "SVC_DT_M"], "left"))

_prov = df_rx.groupBy("PROVIDER_ID").count().orderBy(F.desc("count")).first()["PROVIDER_ID"]
print(f"Example provider: {_prov}")
display(monthly.filter(F.col("PROVIDER_ID") == _prov).orderBy("SVC_DT_M"))

## 3. The original output (`kerendia_nba_3_0_claim_rx_nps_monthly`)
After counting, the original fills months without claims (`step2_fill_missing`), adds the last 1/3/6/12-month sums and `RUNNING_` totals (`step3_lag`), `PCT_` shares (each start column / `TOTAL_PATIENTS_VISIT` in the same window), joins `TOTAL_NEW_PATIENT_START` with its own windows, keeps months from 2022-01 and writes one row per provider per month.

Tables below: row and column counts, columns by type, nulls, and a few columns for the test HCPs' providers.

In [0]:
import pandas as pd
df_orig = spark.read.format("delta").load(_NOTEBOOK_OUTPUT + "kerendia_nba_3_0_claim_rx_nps_monthly")
_orig_cols = df_orig.columns

def _kind(c):
    if c in ("PROVIDER_ID", "SVC_DT_M"): return "key"
    if c.startswith("PCT_"):     return "PCT_"
    if c.startswith("RUNNING_"): return "RUNNING_"
    if "_IN_LAST_" in c:         return "_IN_LAST_N_MONTH"
    return "base month"

def _group(c):
    if "NEW_PATIENT_START" in c: return "Total new patients"
    if "PATIENTS_VISIT" in c:   return "Patients visiting"
    if "PATIENTS_START_" in c:  return "Starts per class / brand"
    return "Keys"

_cols = pd.DataFrame({"column": _orig_cols})
_cols["group"] = _cols["column"].map(_group)
_cols["type"]  = _cols["column"].map(_kind)
print(f"kerendia_nba_3_0_claim_rx_nps_monthly: {df_orig.count():,} rows | {len(_orig_cols)} columns")
display(df_orig.agg(F.min("SVC_DT_M").alias("first_month"), F.max("SVC_DT_M").alias("last_month"),
                    F.countDistinct("PROVIDER_ID").alias("providers")))
display(_cols.pivot_table(index="group", columns="type", values="column", aggfunc="count", fill_value=0).reset_index())

# Columns with nulls
_nulls = df_orig.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in _orig_cols]).first().asDict()
display(pd.DataFrame([(c, n) for c, n in _nulls.items() if n], columns=["column", "null_rows"]))

# Test HCPs' providers
_provs = [r[0] for r in df_rx.select("PROVIDER_ID").distinct().collect()]
_show = ["PROVIDER_ID", "SVC_DT_M", "TOTAL_PATIENTS_VISIT", "TOTAL_PATIENTS_START_SGLT2", "TOTAL_PATIENTS_START_JARDIANCE",
         "TOTAL_NEW_PATIENT_START", "TOTAL_PATIENTS_START_SGLT2_IN_LAST_3_MONTH", "RUNNING_TOTAL_PATIENTS_START_SGLT2",
         "PCT_TOTAL_PATIENTS_START_SGLT2", "PCT_TOTAL_PATIENTS_START_SGLT2_IN_LAST_3_MONTH"]
_show = [c for c in _show if c in _orig_cols]
display(df_orig.filter(F.col("PROVIDER_ID").isin(_provs)).select(*_show).orderBy("PROVIDER_ID", "SVC_DT_M").limit(200))
display(_cols)

## Walkthrough findings (to confirm with Wenjie before the refactor)

Numbers are for the 10 test HCPs unless stated otherwise.

| # | Finding | Evidence | Impact on the refactor |
| --- | --- | --- | --- |
| 1 | **"New" start is left-censored.** The table starts on 2021-01-02, so a patient's first claim *in the data* counts as a start even if they were already on the drug. | First starts per month: 2,630 in 2021-01, about 1,260 in 2021-02/03, then about 200 per month from 2022. Example provider 7608159: 107 starts in 2021-01 vs about 20 per month in 2022. | The output keeps months from 2022-01 (12-month burn-in). Cohorts from 2025-01 have a 4-year burn-in, so they are not affected. |
| 2 | **Starts are looked up across all HCPs.** The first-start date is computed over every claim of the patient, not only the HCP's own. | 16,195 patient-class starts; 34 of them (0.2%) were credited to 2 HCPs (same first day). | The refactor must compute first-start dates on the full table (or on all claims of the run's patients, as done here in validation). |
| 3 | **`Diuretic + MRA` never appears** as a brand group in the table, so there is no `TOTAL_PATIENTS_START_DIURETIC_AND_MRA` column. | 7 of the 8 brands present; 16 start columns in the output (9 classes + 7 brands). | Keep the same 16 columns (fixed pivot values), so every cohort has the same columns. |
| 4 | **Months with no claims are null, not 0.** `step2_fill_missing` adds every month for every provider, but the main spine is not filled with 0 after it (only `TOTAL_NEW_PATIENT_START` is). | Original output: 1,014,149 rows (7.8%) have null base counts; window and `RUNNING_` columns are null until the provider's first claim month (e.g. 676,937 null `RUNNING_` rows). `PCT_` columns become 0.0. | The cohort output has one row per HCP per cohort, so this only matters for HCPs with no claims in a window. The offset-0 comparison must treat null and 0 as equal; confirm 0 is acceptable. |
| 5 | **`TOTAL_NEW_PATIENT_START` is null for providers with no start ever** (it is left-joined at the end). | 5,824 null rows in the original output. | Same treatment as finding 4. |
| 6 | **`PCT_` uses a sum of monthly visit counts as the denominator.** `TOTAL_PATIENTS_VISIT_IN_LAST_3_MONTH` adds monthly distinct patients, so a patient seen in 3 months counts 3 times. | From `step3_lag` (sum of monthly values). | Keep as is (feature logic unchanged). |
| 7 | **Specialty is part of the distinct key**, so an HCP with two specialty codes would count a start twice. | All 10 test HCPs have a single specialty code. | No impact for the test HCPs; keep as is. |
| 8 | **Header text is stale.** It mentions data from 2016/2017 and a 2019-11 filter; the data starts in 2021-01, so these filters do nothing. | `kerendia_nba_3_0_Rx_05`: 4.03B rows, 2021-01-02 to 2026-08-28, 231,346 providers. | None. |

Next: section 4 (cohort features), which uses the same approach as notebook 04: a long table of (provider, date, feature) rows built once, then one pivot per cohort.

## 4. Cohort features (the refactor)

Same feature logic as the original, rebuilt per cohort. First we build a long feature-row table once; then we pivot those rows into one cohort-wide row per provider.

* `build_cohort` (one cohort, reference) filters the rows by date for each window.
* `build_all_cohorts_fast` (used for the output) sums the rows per provider x cohort month x feature, tags each cohort month with every (cohort, window) it belongs to and builds all cohorts in one pivot (same approach as NB01-NB03, shared engine in `00_cohort_config`). The check cell compares the two on the first and last cohort.

In [0]:
import re, time
import pandas as pd

NB05_LOOKBACK_WINDOWS = HCP_LOOKBACK_WINDOWS["nps"]   # original windows (1, 3, 6, 12)
TOTAL_COL            = "TOTAL_PATIENTS_VISIT"
TOTAL_NEW_COL        = "TOTAL_NEW_PATIENT_START"
START_PREFIX         = "TOTAL_PATIENTS_START_"

# Exact NB05 cleaning from the original notebook (cell 17)
_NB05_CLEAN_STEPS = [(", ", " "), ("/", "_OR_"), ("+", "_AND_"), ("-", "_"), (" ", "_"), ("__", "_")]

def clean_name_05(c):
    for a, b in _NB05_CLEAN_STEPS:
        c = F.regexp_replace(c, re.escape(a), b)
    return F.upper(c)

# Fixed feature list from Wenjie's output: 16 start columns (9 classes + 7 brands; Diuretic + MRA absent)
df_orig = spark.read.format("delta").load(_NOTEBOOK_OUTPUT + "kerendia_nba_3_0_claim_rx_nps_monthly")
START_FEATURES = [
    c for c in df_orig.columns
    if c.startswith(START_PREFIX)
    and "_IN_LAST_" not in c
    and not c.startswith("RUNNING_")
    and not c.startswith("PCT_")
]
FEATURE_VALUES = [TOTAL_COL] + START_FEATURES + [TOTAL_NEW_COL]
PCT_FEATURES   = list(START_FEATURES)


# Brand columns present in the original output (7 in practice)
BRAND_FEATURES = [START_PREFIX + clean_col_name(b) for b in _BRANDS if START_PREFIX + clean_col_name(b) in START_FEATURES]


def build_feature_rows(df_rx_in, df_rx_pat_in, offset_days=None):
    """Long feature table: PROVIDER_ID x event date x feature with integer N."""
    rx = df_rx_in.withColumn("SVC_DT_M", assign_cohort_month(F.col(SVC_COL), offset_days))

    # 1) Monthly distinct patients visiting a provider (specialty kept in the distinct key, as in the original)
    visit_rows = (
        rx.select("PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "SVC_DT_M")
          .groupBy("PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", "SVC_DT_M")
          .agg(F.min(SVC_COL).alias(SVC_COL))
          .select("PROVIDER_ID", SVC_COL,
                  F.lit(TOTAL_COL).alias("FEATURE"),
                  F.lit(1).alias("N"))
    )

    # 2) First-start dates across ALL HCPs for the run's patients (all patients in production)
    first_cat = (
        df_rx_pat_in.groupBy("PATIENT_ID", "TREATMENT_CATEGORY")
                   .agg(F.min(SVC_COL).alias("FIRST_SVC_DT"))
    )
    first_brand = (
        df_rx_pat_in.filter(F.col("CATEGORY").isin(_BRANDS))
                   .groupBy("PATIENT_ID", "CATEGORY")
                   .agg(F.min(SVC_COL).alias("FIRST_SVC_DT"))
    )

    # 3) Flagged class starts credited to the provider who wrote the first claim
    class_start_claims = (
        rx.select("PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY", "SVC_DT_M").distinct()
          .join(first_cat.withColumnRenamed("FIRST_SVC_DT", SVC_COL),
                on=["PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY"], how="inner")
    )
    class_rows = class_start_claims.select(
        "PROVIDER_ID", SVC_COL,
        F.concat(F.lit(START_PREFIX), clean_name_05(F.col("TREATMENT_CATEGORY"))).alias("FEATURE"),
        F.lit(1).alias("N")
    )

    # 4) Same for brand starts (8-brand filter from the original)
    brand_start_claims = (
        rx.select("PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "SVC_DT_M", "CATEGORY").distinct()
          .filter(F.col("CATEGORY").isin(_BRANDS))
          .join(first_brand.withColumnRenamed("FIRST_SVC_DT", SVC_COL),
                on=["PATIENT_ID", SVC_COL, "CATEGORY"], how="inner")
    )
    brand_rows = brand_start_claims.select(
        "PROVIDER_ID", SVC_COL,
        F.concat(F.lit(START_PREFIX), clean_name_05(F.col("CATEGORY"))).alias("FEATURE"),
        F.lit(1).alias("N")
    )

    # 5) Total new patients = distinct patients with at least one class start in that cohort month
    total_new_rows = (
        class_start_claims.groupBy("PROVIDER_ID", "PATIENT_ID", "SVC_DT_M")
                         .agg(F.min(SVC_COL).alias(SVC_COL))
                         .select("PROVIDER_ID", SVC_COL,
                                 F.lit(TOTAL_NEW_COL).alias("FEATURE"),
                                 F.lit(1).alias("N"))
    )

    return (visit_rows
            .unionByName(class_rows)
            .unionByName(brand_rows)
            .unionByName(total_new_rows))


# Staging prefix for the shared materialize() in 00_cohort_config (in memory in validation mode,
# _STAGE_PREFIX + name in production).
_STAGE_PREFIX = _NOTEBOOK_OUTPUT + "_tmp_nb05_feature_rows_"


# Complete cohorts only (same rule as NB04)
DATA_MAX_DT   = df_rx_all.agg(F.max(SVC_COL)).first()[0]   # same name as NB01-NB04
COHORT_LIST   = get_hcp_cohort_months()   # common end date of all HCP inputs (00_cohort_config); DATA_MAX_DT kept for parity
COHORT_BOUNDS = {m: get_cohort_bounds(m) for m in COHORT_LIST}
COHORT_IDS    = [COHORT_BOUNDS[m]["cohort_id"] for m in COHORT_LIST]
PREVIEW_COHORT = COHORT_MONTH if COHORT_MONTH in COHORT_LIST else COHORT_LIST[0]

# Team rule (3.1, option B): every count, RUNNING_ included, uses claims from DATA_START_DATE on.
# Only the counted claims are filtered: first-start dates still come from df_rx_pat (full history
# from 2019-11-01), so a patient who started a class before DATA_START_DATE is not a new start later.
# The offset-0 parity keeps the full history (as NB01-NB03).
df_rx_run = df_rx.filter(F.col(SVC_COL) >= F.lit(DATA_START_DATE))

_t0 = time.time()
df_feat = materialize(build_feature_rows(df_rx_run, df_rx_pat, offset_days=OFFSET_DAYS), f"offset{OFFSET_DAYS}")
_N_FEAT = df_feat.count()
print(f"Rx data through {DATA_MAX_DT} -> {len(COHORT_LIST)} complete cohorts: {COHORT_LIST[0]} .. {COHORT_LIST[-1]} | preview cohort: {PREVIEW_COHORT}")
print(f"Feature rows: {_N_FEAT:,} (provider x date x feature) in {time.time() - _t0:,.0f}s")
print(f"Feature columns per window: {len(FEATURE_VALUES)} -> starts: {len(START_FEATURES)}, plus visits + total new patients")

In [0]:
# ---- Display: the long feature-row table (what gets counted per provider/date/feature) ----
_family = (F.when(F.col("FEATURE") == TOTAL_COL, "A. monthly patients visiting")
           .when(F.col("FEATURE") == TOTAL_NEW_COL, "D. total new patients")
           .when(F.col("FEATURE").isin(BRAND_FEATURES), "C. brand starts")
           .when(F.col("FEATURE").isin(START_FEATURES), "B. drug-class starts")
           .otherwise("other"))

display(df_feat.withColumn("FAMILY", _family)
        .groupBy("FAMILY")
        .agg(F.count(F.lit(1)).alias("rows"),
             F.sum("N").alias("patients_counted"),
             F.countDistinct("FEATURE").alias("feature_columns"))
        .orderBy("FAMILY"))

_ex = df_feat.groupBy("PROVIDER_ID", SVC_COL).agg(F.sum("N").alias("n")).orderBy(F.desc("n")).first()
display(df_feat.filter((F.col("PROVIDER_ID") == _ex["PROVIDER_ID"]) & (F.col(SVC_COL) == _ex[SVC_COL]))
        .withColumn("FAMILY", _family)
        .orderBy("FAMILY", "FEATURE"))

In [0]:
# ---- One cohort: windows + RUNNING_ in a single pivot, then PCT_ and metadata ----
def build_cohort(cb, df_feat_in=None, feature_values=None):
    """All features for one cohort, one row per provider with any Rx claim up to the cutoff."""
    L    = df_feat if df_feat_in is None else df_feat_in
    vals = FEATURE_VALUES if feature_values is None else feature_values
    wins = NB05_LOOKBACK_WINDOWS
    cutoff = F.to_date(F.lit(cb["cutoff"].strftime("%Y-%m-%d")))

    # Each row goes to RUNNING_ and to every window whose (start, cutoff] contains its event date
    _suffixes = F.array(F.lit("RUNNING"), *[
        F.when(F.col(SVC_COL) > F.to_date(F.lit(cb["window_bounds"][w]["start"])), F.lit(f"_IN_LAST_{w}_MONTH"))
        for w in wins
    ])

    long = (
        L.filter(F.col(SVC_COL) <= cutoff)
         .withColumn("_suffix", F.explode(_suffixes))
         .filter(F.col("_suffix").isNotNull())
         .withColumn(
             "FEATURE_OUT",
             F.when(F.col("_suffix") == "RUNNING", F.concat(F.lit("RUNNING_"), F.col("FEATURE")))
              .otherwise(F.concat(F.col("FEATURE"), F.col("_suffix")))
         )
    )

    _window_cols  = [f"{f}_IN_LAST_{w}_MONTH" for w in wins for f in vals]
    _running_cols = [f"RUNNING_{f}" for f in vals]
    _pivot_vals   = _window_cols + _running_cols

    wide = (long.groupBy("PROVIDER_ID")
                .pivot("FEATURE_OUT", values=_pivot_vals)
                .agg(F.sum("N")))

    out = wide.na.fill(0)

    pct_exprs = []
    for w in wins:
        denom = f"{TOTAL_COL}_IN_LAST_{w}_MONTH"
        for feat in PCT_FEATURES:
            num = f"{feat}_IN_LAST_{w}_MONTH"
            pct_exprs.append(
                F.when(F.col(denom) != 0, F.col(num) / F.col(denom))
                 .otherwise(F.lit(0.0))
                 .alias(f"PCT_{num}")
            )

    # Same metadata columns as notebooks 02-04
    return (out.select("PROVIDER_ID", *_window_cols, *_running_cols, *pct_exprs)
               .withColumn("COHORT_ID", F.lit(cb["cohort_id"]))
               .withColumn("COHORT_MONTH", F.lit(cb["cohort_month"]))
               .withColumn("OFFSET_DAYS", F.lit(cb["offset_days"]))
               .withColumn("CUTOFF_DATE", F.lit(cb["cutoff"].strftime("%Y-%m-%d")))
               .withColumn("SVC_DT_M", F.to_timestamp(F.lit(f"{cb['cohort_month']}-01")))
               .withColumn("PULL_DT", F.current_date()))


_preview_cb = COHORT_BOUNDS[PREVIEW_COHORT]
_t1 = time.time()
claims_preview = build_cohort(_preview_cb)
_pv_cols = claims_preview.columns
print(f"Preview {_preview_cb['cohort_id']} (cutoff {_preview_cb['cutoff']:%Y-%m-%d}) | providers: {claims_preview.count():,} | columns: {len(_pv_cols):,} | {time.time() - _t1:,.0f}s")

In [0]:
# ---- Display: preview cohort, keys + totals + example columns ----
_ex_cols = [c for c in [
    f"{TOTAL_COL}_IN_LAST_1_MONTH", f"{TOTAL_COL}_IN_LAST_3_MONTH", f"{TOTAL_COL}_IN_LAST_12_MONTH", f"RUNNING_{TOTAL_COL}",
    "TOTAL_PATIENTS_START_SGLT2_IN_LAST_12_MONTH",
    "TOTAL_PATIENTS_START_JARDIANCE_IN_LAST_12_MONTH",
    "TOTAL_NEW_PATIENT_START_IN_LAST_12_MONTH",
    "RUNNING_TOTAL_NEW_PATIENT_START",
    "PCT_TOTAL_PATIENTS_START_SGLT2_IN_LAST_12_MONTH",
    "PCT_TOTAL_PATIENTS_START_JARDIANCE_IN_LAST_12_MONTH",
] if c in _pv_cols]

display(claims_preview.select("PROVIDER_ID", "COHORT_ID", "CUTOFF_DATE", *_ex_cols).orderBy("PROVIDER_ID"))

In [0]:
# ---- Fast engine (pattern A, as NB01-NB03): all cohorts in ONE query ----
# Same columns, order, types and values as build_cohort (checked in the next cell):
# 1) feature rows -> (PROVIDER_ID, cohort month, FEATURE, N), once per feature-row table (materialized).
#    Visits and total new patients are already one row per patient per cohort month, so summing
#    them by cohort month keeps the distinct counts of build_cohort.
# 2) one pivot by provider x cohort through the window tags of 00_cohort_config, then PCT_ and metadata.
OUTPUT_COLS = build_cohort(COHORT_BOUNDS[PREVIEW_COHORT]).columns   # schema only, nothing is computed
_FEAT_MONTHS = {}

def get_feat_months(df_feat_in, offset_days, name):
    if name not in _FEAT_MONTHS:
        _t = time.time()
        _FEAT_MONTHS[name] = materialize(feature_rows_to_months(df_feat_in, "PROVIDER_ID", offset_days), f"months_{name}")
        print(f"Feature rows by cohort month ({name}) built in {time.time() - _t:,.0f}s")
    return _FEAT_MONTHS[name]

def build_all_cohorts_fast(cbs, df_feat_in=None, name="run"):
    """build_cohort for every cohort in cbs, in one query (one row per PROVIDER_ID x COHORT_ID).
    df_feat_in / name: another feature-row table (e.g. the offset-0 parity rows)."""
    L    = df_feat if df_feat_in is None else df_feat_in
    wins = NB05_LOOKBACK_WINDOWS
    off  = cbs[0]["offset_days"]
    wide = build_all_cohorts_from_months(get_feat_months(L, off, f"{name}_offset{off}"), cbs, "PROVIDER_ID",
                                         FEATURE_VALUES, wins)
    pct = []
    for w in wins:
        den = F.col(f"{TOTAL_COL}_IN_LAST_{w}_MONTH")
        pct += [F.when(den != 0, F.col(f"{f}_IN_LAST_{w}_MONTH") / den).otherwise(F.lit(0.0))
                 .alias(f"PCT_{f}_IN_LAST_{w}_MONTH") for f in PCT_FEATURES]
    return add_cohort_meta(wide.select("*", *pct), cbs).select(*OUTPUT_COLS)

print(f"Fast engine ready: {len(OUTPUT_COLS):,} output columns per cohort, {len(COHORT_LIST)} cohorts in one query")

In [0]:
# ---- Check: the all-cohorts query gives exactly what build_cohort gives (first and last cohort) ----
# Nothing is written.
import numpy as np

_check = [COHORT_LIST[0], COHORT_LIST[-1]]
_t0 = time.time()
_pd_fast = build_all_cohorts_fast([COHORT_BOUNDS[m] for m in _check]).toPandas()
print(f"All-cohorts query ({len(_check)} cohorts): {time.time() - _t0:,.0f}s")

_rows = []
for m in _check:
    cb = COHORT_BOUNDS[m]
    _t0 = time.time()
    _one   = build_cohort(cb)
    pd_one = _one.toPandas().set_index("PROVIDER_ID")
    _t_one = time.time() - _t0
    pd_all = _pd_fast[_pd_fast["COHORT_ID"] == cb["cohort_id"]].set_index("PROVIDER_ID")

    same_order = list(pd_one.columns) == list(pd_all.columns)
    same_types = _one.dtypes == build_all_cohorts_fast([cb]).dtypes
    cols = [c for c in pd_one.columns if c in pd_all.columns and c != "PULL_DT"]
    num  = [c for c in cols if pd.api.types.is_numeric_dtype(pd_one[c])]
    txt  = [c for c in cols if c not in num]
    common = sorted(set(pd_one.index) & set(pd_all.index))
    bad = ~np.isclose(pd_one.loc[common, num].astype(float).values, pd_all.loc[common, num].astype(float).values,
                      rtol=1e-9, atol=1e-9, equal_nan=True)
    bad_txt = int((pd_one.loc[common, txt].astype(str).values != pd_all.loc[common, txt].astype(str).values).sum())
    _rows.append({"cohort": cb["cohort_id"], "providers_per_cohort": len(pd_one), "providers_fast": len(pd_all),
                  "providers_common": len(common), "same_column_order": same_order, "same_types": same_types,
                  "cells_compared": int(bad.size) + len(common) * len(txt),
                  "mismatching_cells": int(bad.sum()) + bad_txt, "per_cohort_seconds": builtins.round(_t_one)})

df_fast_check = pd.DataFrame(_rows)
display(df_fast_check)
_ok = (df_fast_check["mismatching_cells"].sum() == 0 and df_fast_check["same_column_order"].all()
       and df_fast_check["same_types"].all()
       and (df_fast_check["providers_per_cohort"] == df_fast_check["providers_common"]).all()
       and (df_fast_check["providers_fast"] == df_fast_check["providers_common"]).all())
print("PASS: fast engine = build_cohort" if _ok else "CHECK: fast engine differs from build_cohort")

## 5. Build and Write All Cohorts
Builds every cohort in `COHORT_LIST` with the fast engine (one query) and writes them in **one write** (`replaceWhere COHORT_ID IN (...)`, so other cohorts and offsets are kept) to `claim_rx_nps_monthly_cohort`. The input already carries `PROVIDER_ID`, so this is the provider-level output directly (no NPI mapping step). Wenjie's `kerendia_nba_3_0_claim_rx_nps_monthly` is not touched.

Outputs below: one line for the write, then the completeness checks (all cohorts present, one row per provider per cohort, no nulls, same columns in every cohort) and the totals per provider and cohort.

In [0]:
OUTPUT_PATH = _NOTEBOOK_OUTPUT + "claim_rx_nps_monthly_cohort"

# All cohorts in one query and ONE write (fast engine, as NB01-NB03). replaceWhere keeps every
# other cohort / offset already in the table.
_ids_sql = ", ".join(f"'{c}'" for c in COHORT_IDS)
_t_all = time.time()
build_all_cohorts_fast([COHORT_BOUNDS[m] for m in COHORT_LIST]).write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("replaceWhere", f"COHORT_ID IN ({_ids_sql})") \
    .partitionBy("COHORT_ID") \
    .save(OUTPUT_PATH)

print(f"{len(COHORT_LIST)} cohorts ({COHORT_IDS[0]} .. {COHORT_IDS[-1]}) written to {OUTPUT_PATH} "
      f"in one write, {(time.time() - _t_all) / 60:,.1f} min")

In [0]:
# Read the written table directly (independent of the preview variables)
df_all_cohorts = spark.read.format("delta").load(OUTPUT_PATH)

_ids_in_table  = sorted(r[0] for r in df_all_cohorts.select("COHORT_ID").distinct().collect())
_missing       = sorted(set(COHORT_IDS) - set(_ids_in_table))
_extra         = sorted(set(_ids_in_table) - set(COHORT_IDS))
df_all_cohorts = df_all_cohorts.filter(F.col("COHORT_ID").isin(COHORT_IDS))
_n_rows  = df_all_cohorts.count()
_n_keys  = df_all_cohorts.select("PROVIDER_ID", "COHORT_ID").distinct().count()
_n_provs = df_all_cohorts.select("PROVIDER_ID").distinct().count()
_nulls   = df_all_cohorts.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_all_cohorts.columns]).first()
_null_cells = builtins.sum(v or 0 for v in _nulls)

# Expected columns: 18 features x 4 windows + 18 RUNNING_ + 16 x 4 PCT_ + 7 metadata (incl. PROVIDER_ID)
_exp_cols = (len(FEATURE_VALUES) * len(NB05_LOOKBACK_WINDOWS) + len(FEATURE_VALUES)
             + len(PCT_FEATURES) * len(NB05_LOOKBACK_WINDOWS) + 7)

print(f"Cohorts expected: {len(COHORT_IDS)} | in table: {len(_ids_in_table)} | "
      f"missing: {_missing or 'none'} | extra (not in this run): {_extra or 'none'}")
print(f"Rows: {_n_rows:,} | distinct (PROVIDER_ID, COHORT_ID): {_n_keys:,} | duplicates: {_n_rows - _n_keys} | "
      f"providers: {_n_provs} | columns: {len(df_all_cohorts.columns):,} (expected {_exp_cols}) | null cells: {_null_cells:,}")
display(df_all_cohorts.groupBy("COHORT_ID", "CUTOFF_DATE").count().orderBy("COHORT_ID"))

_key_cols = ["PROVIDER_ID", "COHORT_ID", "COHORT_MONTH", "OFFSET_DAYS", "CUTOFF_DATE", "SVC_DT_M", "PULL_DT"]
_tot_cols = ([f"{TOTAL_COL}_IN_LAST_{w}_MONTH" for w in NB05_LOOKBACK_WINDOWS] + [f"RUNNING_{TOTAL_COL}"]
             + [f"{TOTAL_NEW_COL}_IN_LAST_{w}_MONTH" for w in NB05_LOOKBACK_WINDOWS] + [f"RUNNING_{TOTAL_NEW_COL}"])
display(df_all_cohorts.select(*_key_cols, *_tot_cols).orderBy("PROVIDER_ID", "COHORT_ID"))

## 6. Validation

1. **Offset = 0 comparison with the original (every cohort):** rebuild the feature rows at offset 0 (calendar months) for the test HCPs and compare every column with Wenjie's `kerendia_nba_3_0_claim_rx_nps_monthly`, cohort month vs original month. Original base-month columns (including base-month `PCT_`) are compared with our `_IN_LAST_1_MONTH` columns; original nulls count as 0 (findings 4 and 5). Providers that are only in the original for a month (no claims yet) must be all zeros.
2. **HCP trace:** for the busiest test HCP, rebuild every feature in pandas from the raw table (separate code: pandas first-start dates, Python name cleaning, windows counted by cohort month instead of dates) and compare every window, `RUNNING_` and `PCT_` value with the written table, for every cohort.

Outputs below: one summary row per month / cohort (cells compared, mismatches) and a mismatch table, which should be empty.

In [0]:
import numpy as np

# ---- Validation 1: offset = 0 parity vs original kerendia_nba_3_0_claim_rx_nps_monthly, every cohort ----
_npis = get_test_npis()
if _npis is None:
    print("Production mode: parity check skipped (runs on the test HCPs only)")
else:
    _t_par = time.time()
    df_feat0 = materialize(build_feature_rows(df_rx, df_rx_pat, offset_days=0), "offset0")
    print(f"Offset-0 feature rows: {df_feat0.count():,} | {time.time() - _t_par:,.0f}s")

    _provs_test = [r[0] for r in df_rx.select("PROVIDER_ID").distinct().collect()]
    _orig_all = (df_orig.filter(F.col("PROVIDER_ID").isin(_provs_test))
                 .withColumn("_M", F.date_format("SVC_DT_M", "yyyy-MM")))
    _orig_months  = {r[0] for r in _orig_all.select("_M").distinct().collect()}
    PARITY_MONTHS = [m for m in get_cohort_months(max_data_date=DATA_MAX_DT, offset_days=0) if m in _orig_months]
    pd_orig_all = _orig_all.filter(F.col("_M").isin(PARITY_MONTHS)).toPandas()
    print(f"Original rows for the test providers: {len(pd_orig_all):,} | months compared: {len(PARITY_MONTHS)}")

    _skip = {"SVC_DT_M", "PULL_DT", "_M", "PROVIDER_ID"}
    _orig_cmp_cols = [c for c in pd_orig_all.columns if c not in _skip]
    # Original column -> cohort-based column (base-month columns map to _IN_LAST_1_MONTH)
    _map = {oc: (oc if ("_IN_LAST_" in oc or oc.startswith("RUNNING_")) else f"{oc}_IN_LAST_1_MONTH")
            for oc in _orig_cmp_cols}
    _meta = {"COHORT_ID", "COHORT_MONTH", "OFFSET_DAYS", "CUTOFF_DATE", "SVC_DT_M", "PULL_DT", "PROVIDER_ID"}

    def _num(df):
        return df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(float)

    # All parity months in ONE query (same engine as the written output), compared month by month
    _cbs0 = {m: get_cohort_bounds(m, offset_days=0) for m in PARITY_MONTHS}
    pd_new_all = build_all_cohorts_fast(list(_cbs0.values()), df_feat0, name="parity").toPandas()

    summary, mismatches = [], []
    for m in PARITY_MONTHS:
        cb0 = _cbs0[m]
        pd_new  = pd_new_all[pd_new_all["COHORT_ID"] == cb0["cohort_id"]].set_index("PROVIDER_ID")
        pd_orig = pd_orig_all[pd_orig_all["_M"] == m].set_index("PROVIDER_ID")

        common    = sorted(set(pd_new.index) & set(pd_orig.index))
        only_orig = sorted(set(pd_orig.index) - set(pd_new.index))
        only_new  = sorted(set(pd_new.index) - set(pd_orig.index))
        present   = [oc for oc in _orig_cmp_cols if _map[oc] in pd_new.columns]
        absent    = [oc for oc in _orig_cmp_cols if _map[oc] not in pd_new.columns]
        extra_new = [c for c in pd_new.columns if c not in _meta and c not in set(_map.values())]

        o = _num(pd_orig.loc[common, present])
        n = _num(pd_new.loc[common, [_map[oc] for oc in present]])
        bad = ~np.isclose(o.values, n.values, rtol=1e-9, atol=1e-9)
        for i, j in list(zip(*np.where(bad)))[:50]:
            mismatches.append({"cohort": cb0["cohort_id"], "PROVIDER_ID": common[i], "original_col": present[j],
                               "new_col": _map[present[j]], "original": o.values[i, j], "new": n.values[i, j]})
        # Providers only in the original that month (no claims yet) must be all zeros
        only_orig_nonzero = (int((_num(pd_orig.loc[only_orig, _orig_cmp_cols]) != 0).any(axis=1).sum())
                             if only_orig else 0)

        summary.append({"cohort_offset0": cb0["cohort_id"], "cutoff": f"{cb0['cutoff']:%Y-%m-%d}",
                        "providers_compared": len(common), "only_in_original": len(only_orig),
                        "only_in_original_nonzero": only_orig_nonzero, "only_in_new": len(only_new),
                        "columns_compared": len(present), "original_cols_missing": len(absent),
                        "new_cols_not_in_original": len(extra_new),
                        "cells_compared": int(o.size), "mismatches": int(bad.sum())})

    df_parity = pd.DataFrame(summary)
    _tot_mis = int(df_parity["mismatches"].sum())
    _ok = (_tot_mis == 0 and df_parity["only_in_original_nonzero"].sum() == 0 and df_parity["only_in_new"].sum() == 0
           and df_parity["original_cols_missing"].sum() == 0)
    print(f"Parity {'PASSED' if _ok else 'FAILED'}: {len(PARITY_MONTHS)} months | "
          f"cells compared: {int(df_parity['cells_compared'].sum()):,} | mismatches: {_tot_mis:,} | "
          f"{time.time() - _t_par:,.0f}s")
    display(df_parity)
    if mismatches:
        display(pd.DataFrame(mismatches))
    else:
        print("Mismatch table: empty (no differing cells)")

In [0]:
# ---- Validation 2: HCP trace. Every feature rebuilt in pandas from the raw table vs written table ----
# Fixed trace HCP (00_cohort_config); its provider from the claims
_trace_row = df_rx.filter(F.col(NPI_COL) == HCP_TRACE_NPIS["nps"]).select(NPI_COL, "PROVIDER_ID").first()
TRACE_NPI, TRACE_PROV = _trace_row[NPI_COL], _trace_row["PROVIDER_ID"]
_off = OFFSET_DAYS
_t_tr = time.time()

def _pm(s):
    """Cohort month in pandas (the offset-th day starts the month)."""
    return ((s - pd.Timedelta(days=_off - 1)) if _off > 0 else s).dt.to_period("M")

def _py_clean(s):
    for a, b in [(", ", " "), ("/", "_OR_"), ("+", "_AND_"), ("-", "_"), (" ", "_"), ("__", "_")]:
        s = s.replace(a, b)
    return s.upper()

_cols_needed = ["PROVIDER_ID", "PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY", "CATEGORY"]
# Raw table, own cleaning (trailing spaces) and own date filter
pdx = df_rx_raw.filter(F.col(NPI_COL) == TRACE_NPI).select(*_cols_needed).toPandas()
_tp = pdx["PATIENT_ID"].dropna().unique().tolist()
pall = (df_rx_raw.filter(F.col("PATIENT_ID").isin(_tp))
        .select("PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY", "CATEGORY").toPandas())
for _d in (pdx, pall):
    _d[SVC_COL] = pd.to_datetime(_d[SVC_COL])
    _d["TREATMENT_CATEGORY"] = _d["TREATMENT_CATEGORY"].str.rstrip()
pdx  = pdx[pdx[SVC_COL] >= DATA_START_DATE]     # counted claims: team rule (RUNNING_ included)
pall = pall[pall[SVC_COL] >= HCP_RX_HISTORY_START]   # full history for first-start dates
pdx["M"] = _pm(pdx[SVC_COL])

# First dates per patient x class and patient x brand, across all HCPs
_first_cat   = pall.groupby(["PATIENT_ID", "TREATMENT_CATEGORY"])[SVC_COL].min().rename("FIRST").reset_index()
_first_brand = (pall[pall["CATEGORY"].isin(_BRANDS)]
                .groupby(["PATIENT_ID", "CATEGORY"])[SVC_COL].min().rename("FIRST").reset_index())

# Events (FEATURE, cohort month), one per +1
_visits = pdx[["PRIMARY_SPECIALTY_CODE", "PATIENT_ID", "M"]].drop_duplicates().assign(FEATURE=TOTAL_COL)
_cs = (pdx[["PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "TREATMENT_CATEGORY", "M"]].drop_duplicates()
       .merge(_first_cat, on=["PATIENT_ID", "TREATMENT_CATEGORY"]))
_cs = _cs[_cs[SVC_COL] == _cs["FIRST"]].copy()
_cs["FEATURE"] = START_PREFIX + _cs["TREATMENT_CATEGORY"].map(_py_clean)
_bs = (pdx[pdx["CATEGORY"].isin(_BRANDS)][["PRIMARY_SPECIALTY_CODE", "PATIENT_ID", SVC_COL, "CATEGORY", "M"]].drop_duplicates()
       .merge(_first_brand, on=["PATIENT_ID", "CATEGORY"]))
_bs = _bs[_bs[SVC_COL] == _bs["FIRST"]].copy()
_bs["FEATURE"] = START_PREFIX + _bs["CATEGORY"].map(_py_clean)
_tn = _cs[["PATIENT_ID", "M"]].drop_duplicates().assign(FEATURE=TOTAL_NEW_COL)
_events = pd.concat([d[["FEATURE", "M"]] for d in (_visits, _cs, _bs, _tn)], ignore_index=True)
_counts = _events.groupby(["FEATURE", "M"]).size()

_written = (spark.read.format("delta").load(OUTPUT_PATH)
            .filter((F.col("PROVIDER_ID") == TRACE_PROV) & F.col("COHORT_ID").isin(COHORT_IDS)).toPandas()
            .set_index("COHORT_ID"))

trace_rows, trace_bad = [], []
for m in COHORT_LIST:
    cb = COHORT_BOUNDS[m]
    Mc = pd.Period(m, "M").ordinal
    exp = {}
    for f in FEATURE_VALUES:
        s = _counts.xs(f, level="FEATURE") if f in _counts.index.get_level_values("FEATURE") else pd.Series(dtype=int)
        mi = np.array([p.ordinal for p in s.index], dtype=int)  # month as integer ordinal
        v  = s.to_numpy()
        exp[f"RUNNING_{f}"] = int(v[mi <= Mc].sum())
        for w in NB05_LOOKBACK_WINDOWS:
            exp[f"{f}_IN_LAST_{w}_MONTH"] = int(v[(mi > Mc - w) & (mi <= Mc)].sum())
    for w in NB05_LOOKBACK_WINDOWS:
        den = exp[f"{TOTAL_COL}_IN_LAST_{w}_MONTH"]
        for f in PCT_FEATURES:
            exp[f"PCT_{f}_IN_LAST_{w}_MONTH"] = (exp[f"{f}_IN_LAST_{w}_MONTH"] / den) if den else 0.0
    row = _written.loc[cb["cohort_id"]] if cb["cohort_id"] in _written.index else None
    n_bad = 0
    for c, v in exp.items():
        got = float(row[c]) if row is not None and c in row.index else None
        if got is None or not np.isclose(got, v, rtol=1e-9, atol=1e-9):
            n_bad += 1
            if len(trace_bad) < 50:
                trace_bad.append({"cohort": cb["cohort_id"], "column": c, "expected": v, "written": got})
    trace_rows.append({"cohort": cb["cohort_id"], "cutoff": f"{cb['cutoff']:%Y-%m-%d}", "checks": len(exp),
                       "mismatches": n_bad,
                       "visits_12m": exp[f"{TOTAL_COL}_IN_LAST_12_MONTH"],
                       "new_patients_12m": exp[f"{TOTAL_NEW_COL}_IN_LAST_12_MONTH"]})

df_trace = pd.DataFrame(trace_rows)
_n_checks, _n_bad = int(df_trace["checks"].sum()), int(df_trace["mismatches"].sum())
print(f"HCP trace {'PASSED' if _n_bad == 0 else 'FAILED'}: NPI {TRACE_NPI} / provider {TRACE_PROV} | "
      f"claims {len(pdx):,} | patients {len(_tp):,} | all claims of those patients {len(pall):,} | "
      f"cohorts {len(COHORT_LIST)} | checks {_n_checks:,} | mismatches {_n_bad:,} | {time.time() - _t_tr:,.0f}s")
display(df_trace)
if trace_bad:
    display(pd.DataFrame(trace_bad))
else:
    print("Mismatch table: empty (no differing values)")